# IoT Dataset Analysis Demo — Intel Berkeley Research Lab

**Instructor demo for EECE5155 Hands-On Session 3**  
**Date:** March 26, 2026

This notebook demonstrates the three analysis exercises on the Intel Lab dataset.  
Students will then apply the same methods to other datasets as homework.

---

**Dataset:** Intel Berkeley Research Lab (2004)  
**Source:** https://db.csail.mit.edu/labdata/labdata.html  
**Size:** ~2.3 million readings from 54 sensors over 38 days

## Step 0: Load the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

# Download if not present
import os
import urllib.request
import gzip

data_path = 'data.txt'
if not os.path.exists(data_path):
    print('Downloading Intel Lab dataset...')
    url = 'https://db.csail.mit.edu/labdata/data.txt.gz'
    gz_path = 'data.txt.gz'
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, 'rb') as f_in:
        with open(data_path, 'wb') as f_out:
            f_out.write(f_in.read())
    print('Download complete.')

In [ ]:
# Load data - space-separated, no headers
columns = ['date', 'time', 'epoch', 'moteid', 'temperature', 'humidity', 'light', 'voltage']
df = pd.read_csv(data_path, sep=r'\s+', names=columns, na_values=[''])

# Create timestamp column
df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')

# Drop rows with invalid timestamps
df = df.dropna(subset=['timestamp'])

# Filter to valid mote IDs (1-54)
df = df[df['moteid'].between(1, 54)]

print(f'Loaded {len(df):,} rows')
print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Unique sensors: {df["moteid"].nunique()}')
df.head()

In [ ]:
# Check missing values
print('Missing values per column:')
df.isnull().sum()

---
## Exercise 1: Data Quality Audit

**Questions:**
1. How many readings does each sensor have? Which sensors have very few readings?
2. What is the distribution of temperature readings? Are there outliers?
3. Plot temperature over time for a few sensors. Do any sensors stop reporting?
4. What percentage of temperature readings are above 40°C? Is this physically plausible?

In [ ]:
# 1. Readings per sensor
readings_per_mote = df.groupby('moteid').size().sort_values(ascending=False)

print('Top 5 sensors by readings:')
print(readings_per_mote.head())
print('\nBottom 10 sensors by readings:')
print(readings_per_mote.tail(10))

# Histogram of readings per sensor
fig, ax = plt.subplots()
readings_per_mote.hist(bins=30, ax=ax)
ax.set_xlabel('Number of readings')
ax.set_ylabel('Number of sensors')
ax.set_title('Distribution of readings per sensor')
plt.savefig('audit_readings_per_sensor.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Temperature distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full distribution
df['temperature'].hist(bins=100, ax=axes[0])
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Count')
axes[0].set_title('Temperature distribution (full range)')
axes[0].axvline(x=40, color='red', linestyle='--', label='40°C threshold')
axes[0].legend()

# Zoomed to plausible range
df['temperature'].hist(bins=100, ax=axes[1], range=(15, 35))
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Count')
axes[1].set_title('Temperature distribution (plausible range: 15-35°C)')

plt.tight_layout()
plt.savefig('audit_temperature_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Percentage above 40°C
total_temp_readings = df['temperature'].notna().sum()
above_40 = (df['temperature'] > 40).sum()
pct_above_40 = 100 * above_40 / total_temp_readings

print(f'Total temperature readings: {total_temp_readings:,}')
print(f'Readings above 40°C: {above_40:,} ({pct_above_40:.1f}%)')
print(f'\nInterpretation: {pct_above_40:.1f}% of readings are above 40°C in a lab maintained at 20-22°C.')
print('These are clearly sensor errors (ADC glitches, electrical noise, or stuck readings).')

In [ ]:
# 4. Time series for selected sensors
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for i, mote_id in enumerate([1, 49, 5]):
    mote_data = df[df['moteid'] == mote_id].sort_values('timestamp')
    axes[i].plot(mote_data['timestamp'], mote_data['temperature'], 'b-', alpha=0.5, markersize=1)
    axes[i].set_ylabel(f'Mote {mote_id}\nTemp (°C)')
    axes[i].set_ylim(15, 50)
    
    # Mark end of data for mote 49
    if mote_id == 49:
        last_valid = mote_data['timestamp'].max()
        axes[i].axvline(x=last_valid, color='red', linestyle='--', alpha=0.7)
        axes[i].text(last_valid, 45, f'Stops here\n{last_valid.date()}', fontsize=9, color='red')

axes[-1].set_xlabel('Timestamp')
axes[0].set_title('Temperature time series for selected sensors')

plt.tight_layout()
plt.savefig('audit_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

### Exercise 1: Key Findings

**Write your interpretation here:**

1. **Sensor reliability:** Motes with <100 readings likely failed early (battery, radio, or physical damage). Mote 5 has only 35 readings — probably died within first few hours.

2. **Temperature outliers:** 18% of readings are above 40°C in a 20-22°C lab. These are sensor errors, not real temperatures. Causes could include:
   - ADC glitches (electromagnetic interference from lab equipment)
   - Stuck ADC registers returning maximum value
   - Electrical noise on the sensor line

3. **Mote 49 failure:** Stops reporting around March 10-12. The pattern before death shows normal readings, suggesting sudden failure (battery depletion or radio failure) rather than gradual degradation.

4. **Key insight:** Without ever visiting the lab, we can identify which sensors were reliable, which failed, when they failed, and what type of failures occurred.

---
## Exercise 2: Compare Sensors

**Questions:**
1. Do nearby sensors agree? Compute correlation matrix for temperature.
2. What is the spread (max - min) between sensors at each timestamp?
3. If you wrote "accuracy ±0.5°C" in your PES, what does this exercise tell you about validating that claim?

In [ ]:
# Resample to 5-minute intervals to reduce noise and align timestamps
df_resampled = df.set_index('timestamp').groupby('moteid').resample('5min').mean(numeric_only=True).reset_index()

# Pivot to get sensors as columns
temp_pivot = df_resampled.pivot(index='timestamp', columns='moteid', values='temperature')

# Keep only sensors with enough data
valid_sensors = temp_pivot.count()[temp_pivot.count() > 1000].index
temp_pivot = temp_pivot[valid_sensors]

print(f'Analyzing {len(valid_sensors)} sensors with sufficient data')

In [ ]:
# 1. Correlation matrix for first 10 sensors
corr_matrix = temp_pivot.iloc[:, :10].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0.8, vmax=1.0, ax=ax)
ax.set_title('Temperature correlation between sensors (Motes 1-10)')
plt.tight_layout()
plt.savefig('compare_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Resample to 5-minute intervals to reduce noise and align timestamps
df_resampled = df.set_index('timestamp').groupby('moteid').resample('5min').mean(numeric_only=True).reset_index()

# Pivot to get sensors as columns
temp_pivot = df_resampled.pivot(index='timestamp', columns='moteid', values='temperature')

# Keep only sensors with enough data
valid_sensors = temp_pivot.count()[temp_pivot.count() > 1000].index
temp_pivot = temp_pivot[valid_sensors]

print(f'Analyzing {len(valid_sensors)} sensors with sufficient data')

In [ ]:
# 3. Compare specific sensor pairs
pairs_to_compare = [(1, 2), (1, 34), (2, 34)]

for m1, m2 in pairs_to_compare:
    if m1 in temp_pivot.columns and m2 in temp_pivot.columns:
        corr = temp_pivot[m1].corr(temp_pivot[m2])
        print(f'Correlation Mote {m1} vs Mote {m2}: {corr:.3f}')

### Exercise 2: Key Findings

**Write your interpretation here:**

1. **Correlation patterns:** Motes 1 and 2 have correlation ~0.98-0.99 — they're physically adjacent in the lab. Mote 34 shows slightly lower correlation (~0.94) with others, likely due to physical distance from the cluster.

2. **Spread analysis:** The max spread of ~120°C is driven by outlier readings (sensor errors). The median spread of <1°C is the real measure of sensor agreement. This shows that when sensors work correctly, they agree within about 1°C.

3. **PES implications:** If you wrote "accuracy ±0.5°C" in your PES, this exercise shows you need a reference sensor to validate that claim. The 1°C median spread suggests that's a more realistic accuracy estimate for this deployment. For your MVP, you need to decide: accuracy against what reference?

---
## Exercise 3: Anomaly Detection

**Questions:**
1. Apply a fixed threshold (flag readings outside 15-35°C). How many anomalies?
2. Apply a rolling z-score (flag |z| > 3). How many anomalies?
3. How many anomalies are caught by both methods? Why are the numbers so different?
4. For your MVP: which method for safety alerts? Which for drift detection?

In [ ]:
# Focus on Mote 1 temperature
mote1 = df[df['moteid'] == 1].sort_values('timestamp').copy()
mote1 = mote1.set_index('timestamp')

# Method 1: Fixed threshold
LOWER, UPPER = 15, 35
mote1['anomaly_fixed'] = (mote1['temperature'] < LOWER) | (mote1['temperature'] > UPPER)

# Method 2: Rolling z-score (1-hour window)
window = '1h'
rolling_mean = mote1['temperature'].rolling(window, center=True).mean()
rolling_std = mote1['temperature'].rolling(window, center=True).std()
mote1['z_score'] = (mote1['temperature'] - rolling_mean) / rolling_std
mote1['anomaly_zscore'] = mote1['z_score'].abs() > 3

# Count anomalies
fixed_count = mote1['anomaly_fixed'].sum()
zscore_count = mote1['anomaly_zscore'].sum()  # NaN excluded automatically
both_count = (mote1['anomaly_fixed'] & mote1['anomaly_zscore']).sum()

print(f'Fixed threshold anomalies: {fixed_count:,}')
print(f'Rolling z-score anomalies: {zscore_count:,}')
print(f'Caught by both methods: {both_count:,}')
print(f'\nOverlap: {100*both_count/min(fixed_count, zscore_count):.1f}% of the smaller count')

In [ ]:
# Visualize anomalies
fig, ax = plt.subplots(figsize=(14, 5))

# Plot all readings
ax.plot(mote1.index, mote1['temperature'], 'b-', alpha=0.3, label='Temperature', markersize=1)

# Fixed threshold anomalies
fixed_anomalies = mote1[mote1['anomaly_fixed']]
ax.scatter(fixed_anomalies.index, fixed_anomalies['temperature'], 
           c='red', s=10, alpha=0.5, label=f'Fixed threshold ({fixed_count:,})')

# Z-score anomalies (only those NOT caught by fixed threshold)
zscore_only = mote1[mote1['anomaly_zscore'] & ~mote1['anomaly_fixed']]
ax.scatter(zscore_only.index, zscore_only['temperature'],
           c='green', s=20, alpha=0.7, label=f'Z-score only ({len(zscore_only):,})', marker='^')

# Threshold lines
ax.axhline(y=15, color='orange', linestyle='--', alpha=0.5, label='Fixed thresholds')
ax.axhline(y=35, color='orange', linestyle='--', alpha=0.5)

ax.set_xlabel('Timestamp')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Mote 1: Anomaly Detection Comparison')
ax.legend(loc='upper right')
ax.set_ylim(10, 50)

plt.tight_layout()
plt.savefig('anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Example: anomaly caught by z-score but NOT by fixed threshold
zscore_only_examples = zscore_only.head(5)
print('Examples caught by z-score but NOT fixed threshold:')
print(zscore_only_examples[['temperature', 'z_score']])
print('\nThese are within 15-35°C but statistically unusual (sudden jumps).')

In [ ]:
# Example: anomaly caught by fixed threshold but NOT by z-score
fixed_only = mote1[mote1['anomaly_fixed'] & ~mote1['anomaly_zscore']]
fixed_only_examples = fixed_only.head(5)
print('Examples caught by fixed threshold but NOT z-score:')
print(fixed_only_examples[['temperature', 'z_score']])
print('\nThese are outside 15-35°C but the rolling mean has adapted to consider them "normal".')

### Exercise 3: Key Findings

**Write your interpretation here:**

1. **Method comparison:** Fixed threshold catches ~6,700 anomalies, z-score catches ~500. Only ~10 overlap. This huge difference shows the methods detect fundamentally different things.

2. **Fixed threshold catches:** All readings outside 15-35°C, including sensor error spikes. Good for detecting obvious malfunctions and PES violations. Bad for detecting gradual drift or unusual patterns within range.

3. **Z-score catches:** Sudden jumps regardless of absolute value. Good for detecting unusual events and gradual drift. Bad for constant high readings (rolling mean adapts).

4. **For MVP:**
   - **Fixed thresholds** for safety alerts (PES compliance, regulatory requirements)
   - **Z-score** for drift detection (data quality monitoring, predictive maintenance)
   - Use BOTH for comprehensive coverage

---
## Summary

This demo showed the three core analysis techniques for IoT sensor data:

1. **Quality Audit:** Identify dead sensors, outliers, and data quality issues
2. **Sensor Comparison:** Validate readings against nearby sensors, understand accuracy limits
3. **Anomaly Detection:** Choose appropriate methods for different use cases

**Homework:** Apply these same three exercises to one of the following datasets:
- Google Smart Buildings Dataset (2024) — TensorFlow Datasets
- Numenta Anomaly Benchmark (NAB) — labeled anomalies for validation
- ASHRAE Energy — building HVAC patterns

Submit your LaTeX report with interpretations for each exercise.